# TP2 — RAG avec Ray (INF8810)

**Membres de l'équipe :**
- Membre 1 : Barry Fatima
- Membre 2 : Barry Mamadou Rafiou
- Membre 3 : Ouazar Naouel

---

Dans ce notebook, on reprend le pipeline RAG de base et on utilise **Ray** pour paralléliser les deux parties les plus lentes : le chunking et le calcul des embeddings.

Les autres fonctions (index, recherche, LLM) restent exactement comme dans le RAG de base, on les réutilise telles quelles.

## Avant de lancer Run All

Avant de lancer **Run all**, s'assurer que les librairies suivantes sont installées
(les cellules `pip install` ci-dessous s'en chargent automatiquement) :
- `ray`
- `faiss-cpu`
- `sentence-transformers`
- `transformers`
- `datasets`.

## Installation

In [ ]:
!pip install ray faiss-cpu sentence-transformers transformers datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.9 MB/s eta 0:00:00


## Imports

In [ ]:
import ray
import numpy as np
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

---
## Étape 0 — Initialisation de Ray

On lance Ray avec `ray.init()`. Sur Colab, ça crée un cluster local qui utilise les CPU disponibles.

In [ ]:
ray.init()

2026-05-01 03:08:39,281	INFO worker.py:2012 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.13
Ray version:,2.55.1


---
## Chargement des données

On charge les 100 premiers articles Wikipedia (Simple English), pareil que dans le RAG de base.

In [ ]:
ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train[:100]")
print(f"Articles chargés : {len(ds)}")
print("Exemple :", ds[54]['text'][:100])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

20231101.simple/train-00000-of-00001.par(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/241787 [00:00<?, ? examples/s]

Articles chargés : 100
Exemple : Botany is the study of plants. It is a science. It is a branch of biology, and is also called plant 


---
## Fonction chunk_text

Même fonction que dans le RAG de base, pas de changement ici. Elle coupe le texte en morceaux par paragraphe avec une taille max de 500 caractères.

In [ ]:
def chunk_text(text, max_len=500):
    paras = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    for p in paras:
        if len(p) <= max_len:
            chunks.append(p)
        else:
            words = p.split()
            curr = []
            for w in words:
                if len(' '.join(curr + [w])) <= max_len:
                    curr.append(w)
                else:
                    chunks.append(' '.join(curr))
                    curr = [w]
            if curr:
                chunks.append(' '.join(curr))
    return chunks

---
## Étape 1 — Distribution du chunking avec Ray

Dans le RAG de base, le chunking se faisait de façon séquentielle :
```python
results = [process_doc(doc) for doc in ds]
```

Avec Ray, on ajoute `@ray.remote` sur la fonction et on lance tous les documents en parallèle avec `.remote()`, puis on récupère les résultats avec `ray.get()`.

### Tâche 1a — on ajoute @ray.remote pour rendre process_doc distribuée

In [ ]:
@ray.remote
def process_doc(doc):
    doc_chunks = chunk_text(doc['text'])
    return [(chunk, {'title': doc['title'], 'url': doc['url']}) for chunk in doc_chunks]

### Tâche 1b — lancement en parallèle avec .remote()

In [ ]:
taches = [process_doc.remote(doc) for doc in ds]
print(f"{len(taches)} documents envoyés en parallèle à Ray")

100 documents envoyés en parallèle à Ray


### Tâche 1c — récupération des résultats avec ray.get() puis on sépare chunks et metadata


In [ ]:
docs_traites = ray.get(taches)

chunks   = [chunk for res in docs_traites for chunk, meta in res]
metadata = [meta  for res in docs_traites for chunk, meta in res]

print(f"Chunks obtenus : {len(chunks)}")
print(f"Metadata : {len(metadata)}")
print("Premier chunk :", chunks[0][:80], "...")

Chunks obtenus : 1881
Metadata : 1881
Premier chunk : April (Apr.) is the fourth month of the year in the Julian and Gregorian calenda ...


---
## Étape 2 — Distribution du calcul des embeddings avec Ray

Dans le RAG de base, tous les embeddings se calculaient en un seul appel :
```python
embeddings = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
```

Pour paralléliser ça avec Ray, on divise d'abord les chunks en batches de 50, ensuite on encode chaque batch en parallèle, et on recolle tout avec `np.vstack()`.

### chargement du modèle (identique au RAG de base)


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Tâche 2a — on découpe les chunks en batches de 50


In [ ]:
batch_size = 50
batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
print(f"Nombre de batches : {len(batches)}")

Nombre de batches : 38


### Tâche 2b — version distribuée de encode_batch


In [ ]:
@ray.remote
def encode_batch(chunks_batch):
    # chaque worker charge son propre modèle pour eviter les conflits entre tâches Ray
    emb_model = SentenceTransformer('all-MiniLM-L6-v2')
    return emb_model.encode(chunks_batch, convert_to_numpy=True)

### Tâche 2c — on lance l'encodage de tous les batches en parallèle


In [ ]:
embedding_refs = [encode_batch.remote(batch) for batch in batches]
print(f"{len(embedding_refs)} tâches d'encodage lancées")

38 tâches d'encodage lancées


### Tâche 2d — récupération et assemblage final


In [ ]:
embedding_batches = ray.get(embedding_refs)
embeddings_ray = np.vstack(embedding_batches)
print(f"Shape embeddings : {embeddings_ray.shape}")

(encode_batch pid=3238) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2389.82it/s, Materializing param=pooler.dense.weight]
(encode_batch pid=3238) BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
(encode_batch pid=3238) Key                     | Status     |  | 
(encode_batch pid=3238) ------------------------+------------+--+-
(encode_batch pid=3238) embeddings.position_ids | UNEXPECTED |  | 
(encode_batch pid=3238) 
(encode_batch pid=3238) Notes:
(encode_batch pid=3238) - UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
(encode_batch pid=3237) 
(encode_batch pid=3237) 
(encode_batch pid=3237) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|█████████

Shape embeddings : (1881, 384)


### Tâche 2e — vérification que la shape est correcte (doit être nb_chunks x 384)


In [ ]:
assert embeddings_ray.shape[0] == len(chunks), \
    f"Erreur : {embeddings_ray.shape[0]} embeddings pour {len(chunks)} chunks"
assert embeddings_ray.shape[1] == 384, \
    f"Erreur : dimension attendue 384, obtenue {embeddings_ray.shape[1]}"
print(f"OK — shape : {embeddings_ray.shape}")
print(f"   {embeddings_ray.shape[0]} chunks x {embeddings_ray.shape[1]} dimensions")

OK — shape : (1881, 384)
   1881 chunks x 384 dimensions


---
## Étape 3 — Vérification du pipeline complet

On reprend les fonctions du RAG de base et on les branche sur les chunks et embeddings produits par Ray pour vérifier que tout fonctionne.

### fonctions RAG de base

In [ ]:


def build_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

def search(query, k=5):
    q_vec = model.encode([query], convert_to_numpy=True)
    dists, idxs = index.search(q_vec, k)
    return [(chunks[i], metadata[i], dists[0][j]) for j, i in enumerate(idxs[0])]

def create_prompt(query, contexte):
    context = '\n\n'.join(
        [f"Source: {meta['title']}\n{chunk}" for chunk, meta, dist in contexte]
    )
    return (
        f"Answer the question based on the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )

def call_llm(prompt, max_len=150):
    inputs = llm_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = llm_model.generate(**inputs, max_length=max_len)
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

### Tâche 3a — construction de l'index FAISS avec les embeddings produits par Ray


In [ ]:
index = build_index(embeddings_ray)
print(f"Index construit — {index.ntotal} vecteurs indexés")

Index construit — 1881 vecteurs indexés


### Tâche 3b — test de recherche


In [ ]:
query = "What happened in April 1912?"
res = search(query)

print(f"Requête : {query}\n")
for i, (chunk, meta, dist) in enumerate(res):
    print(f"[{i+1}] {meta['title']}  (distance : {dist:.4f})")
    print(f"     {chunk[:100]}...")
    print()

Requête : What happened in April 1912?

[1] April  (distance : 0.9420)
     Events in April...

[2] December  (distance : 0.9598)
     Historical Events...

[3] April  (distance : 1.0202)
     people on board. April 16, 1943 - Albert Hofmann discovers LSD's effects. April 17, 1946 - Syria gai...

[4] April  (distance : 1.0210)
     space. April 14, 1865 - US President Abraham Lincoln is shot dead at Ford's Theatre by John Wilkes B...

[5] December  (distance : 1.1090)
     previously no human had ever been. December 15, 1891 - James Naismith introduces basketball. Decembe...



### Tâche 3c — chargement du LLM flan-t5-base


In [ ]:
llm_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Tâche 3d — génération de la réponse finale


In [ ]:
query   = "What happened in April 1912?"
res_finaux = search(query)
prompt  = create_prompt(query, res_finaux)

print("=== PROMPT ===")
print(prompt)
print()

reponse = call_llm(prompt)
print("=== RÉPONSE ===")
print(reponse)

=== PROMPT ===
Answer the question based on the context below.

Context:
Source: April
Events in April

Source: December
Historical Events

Source: April
people on board. April 16, 1943 - Albert Hofmann discovers LSD's effects. April 17, 1946 - Syria gains full independence from France. April 17, 1955 - Albert Einstein dies. April 18, 1906 - 1906 San Francisco earthquake: San Francisco, California, is hit by a big earthquake, resulting in fires that destroy large parts of the city. April 18, 1980 - Zimbabwe gains full independence. April 19, 1897 - The first Boston Marathon is held. April 19, 1971 - Sierra Leone becomes a republic. April 19, 1993

Source: April
space. April 14, 1865 - US President Abraham Lincoln is shot dead at Ford's Theatre by John Wilkes Booth. Lincoln dies the next day. April 14, 2010 - Qinghai Province, China, is hit by an earthquake, killing tens of thousands of people. April 14, 2010 - The eruption of Eyjafjallajokull in Iceland shuts down air traffic around Eu

---
## Conclusion

Le changement principal entre le RAG de base et cette version est l'ajout de `@ray.remote` sur les deux fonctions coûteuses, ce qui permet de les exécuter en parallèle au lieu de séquentiellement.

| Étape | RAG de base | Avec Ray |
|-------|-------------|----------|
| Chunking | `[process_doc(doc) for doc in ds]` | `ray.get([process_doc.remote(doc) for doc in ds])` |
| Embeddings | `model.encode(chunks)` | `ray.get([encode_batch.remote(b) for b in batches])` |
| Index / Recherche / LLM | inchangés | inchangés |

Les modifications sont minimales : `@ray.remote` sur la fonction + `.remote()` à l'appel + `ray.get()` pour récupérer. Le reste du pipeline n'a pas bougé.